In [ ]:
!pip install kaggle -q
# Configuring the path of kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# API to fetch the dataset from kaggle
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign

In [ ]:
# Extract the compressed Dataset
from zipfile import ZipFile
import os

zip_path = "/content/gtsrb-german-traffic-sign.zip"

with ZipFile(zip_path, 'r') as zip:
  zip.extractall()

print("!!!Extracted!!!")


# **Step 1️⃣ Define paths**

In [ ]:
test_dir = "/content/Test"
train_dir = "/content/Train"
img_size = (32,32)
batch_size = 32

# **Step 2️⃣ Load images using ImageDataGenerator**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale = 1./255,
    validation_split = 0.2
)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size = img_size,
    batch_size = batch_size,
    class_mode = "categorical",
    subset = "training"
)

val_data = train_gen.flow_from_directory(
    train_dir,
    target_size = img_size,
    batch_size = batch_size,
    class_mode = "categorical",
    subset = "validation"
)

In [ ]:
import pandas as pd

test_df = pd.read_csv("/content/Test.csv")
print(test_df.head())


# **Step 3️⃣ Load test data (no labels used during training)**

In [ ]:
test_gen = ImageDataGenerator(rescale=1./255)

test_df['ClassId'] = test_df['ClassId'].astype(str)

test_data = test_gen.flow_from_dataframe(
    dataframe=test_df,
    directory="/content/",
    x_col="Path",        # image filename column
    y_col="ClassId",     # label column
    target_size=(32, 32),
    batch_size=32,
    class_mode="categorical", # Changed from "sparse" to "categorical"
    shuffle=False # Added shuffle=False to ensure order for evaluation
)

# **Step 4️⃣ CNN Model**

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(train_data.num_classes, activation="softmax")
])


# **Step 5️⃣ Compile & Train**

In [ ]:
model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

model.summary()

history = model.fit(
    train_data,
    validation_data = val_data,
    epochs = 15
)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy: ",test_acc)

In [ ]:
import numpy as np

pred_probs = model.predict(val_data)
pred_classes = np.argmax(pred_probs, axis=1)

print("Predicted classes:", pred_classes[:10])
print("True classes:", val_data.classes[:10])


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/Test/00004.png"

# Load the image with the correct target_size (32,32) to match the model's input
img = image.load_img(img_path, target_size=(32,32))
img = image.img_to_array(img)
img = img / 255.0
img = np.expand_dims(img, axis=0)

prediction = model.predict(img)
predicted_class = np.argmax(prediction)

class_labels = list(train_data.class_indices.keys())

print("Predicted label:", class_labels[predicted_class])